# Maryland Education Data Preparation

This notebook loads, inspects, cleans, and combines Maryland State Department of Education data for 2023–2025.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


## Project Paths and File Organization

Raw files are stored by year under `data/raw/`, while cleaned outputs will be written to `data/processed/`.

In [2]:
PROJECT_ROOT = Path.home() / "Documents" / "Maryland Education Project"

DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(DATA_DIR)

C:\Users\elsie\Documents\Maryland Education Project
C:\Users\elsie\Documents\Maryland Education Project\data


In [3]:
DATA_2023 = RAW_DATA_DIR / "2023"
DATA_2024 = RAW_DATA_DIR / "2024"
DATA_2025 = RAW_DATA_DIR / "2025"

In [4]:
def list_files(folder):
    for path in folder.rglob("*"):
        if path.is_file():
            print(path.relative_to(PROJECT_ROOT))

In [5]:
list_files(RAW_DATA_DIR)

data\raw\2023\2023 MCAP ELA Administrative Data Report Card.xlsx
data\raw\2023\2023 MCAP MATH Administrative Data Report Card.xlsx
data\raw\2023\2023_Special_Services.csv
data\raw\2023\Attendance_2023.csv
data\raw\2024\2024 MCAP ELA Administrative Data Report Card.xlsx
data\raw\2024\2024 MCAP MATH Administrative Data Report Card.xlsx
data\raw\2024\2024_Special_Services.csv
data\raw\2024\Attendance_2024.csv
data\raw\2025\2025 MCAP ELA Administrative Data Report Card.xlsx
data\raw\2025\2025 MCAP MATH Administrative Data Report Card.xlsx
data\raw\2025\2025_Special_Services.csv
data\raw\2025\Attendance_2025.csv


## Load Raw 2025 Data

The 2025 files are loaded first to inspect their structure and develop a cleaning workflow that can later be applied to 2023 and 2024.

In [6]:
math_2025_file = DATA_2025 / "2025 MCAP MATH Administrative Data Report Card.xlsx"
ela_2025_file = DATA_2025 / "2025 MCAP ELA Administrative Data Report Card.xlsx"
attendance_2025_file = DATA_2025 / "Attendance_2025.csv"
special_2025_file = DATA_2025 / "2025_Special_Services.csv"

In [7]:
# load 2025 data files
math_2025 = pd.read_excel(
    math_2025_file,
    sheet_name="School_Level"
)

ela_2025 = pd.read_excel(
    ela_2025_file,
    sheet_name="School_Level"
)

attendance_2025 = pd.read_csv(attendance_2025_file)
special_2025 = pd.read_csv(special_2025_file)

In [8]:
print("Math shape:", math_2025.shape)
print("ELA shape:", ela_2025.shape)
print("Attendance shape:", attendance_2025.shape)
print("Special Services shape:", special_2025.shape)

display(math_2025.head())
display(ela_2025.head())
display(attendance_2025.head())
display(special_2025.head())

Math shape: (5942, 15)
ELA shape: (5253, 15)
Attendance shape: (1816, 18)
Special Services shape: (1826, 30)


,Year,LEA,LEA Name,School,School Name,Assessment,Student Group,Tested Count,Proficient Count,Proficient Pct,Level 1 Pct,Level 2 Pct,Level 3 Pct,Level 4 Pct,Create Date
0,2025,1.0,Allegany,301.0,Flintstone Elementary,Mathematics 03,All Students,34,14,41.2,17.6,41.2,38.2,<= 5.0,20250808.0
1,2025,1.0,Allegany,301.0,Flintstone Elementary,Mathematics 04,All Students,29,13,44.8,20.7,34.5,37.9,6.9,20250808.0
2,2025,1.0,Allegany,301.0,Flintstone Elementary,Mathematics 05,All Students,23,4,17.4,13.0,69.6,17.4,<= 5.0,20250808.0
3,2025,1.0,Allegany,301.0,Flintstone Elementary,Mathematics All Grades,All Students,86,31,36.0,17.4,46.5,32.6,<= 5.0,20250808.0
4,2025,1.0,Allegany,401.0,South Penn Elementary,Mathematics 03,All Students,67,21,31.3,37.3,31.3,31.3,<= 5.0,20250808.0


,Year,LEA,LEA Name,School,School Name,Assessment,Student Group,Tested Count,Proficient Count,Proficient Pct,Level 1 Pct,Level 2 Pct,Level 3 Pct,Level 4 Pct,Create Date
0,2025,1.0,Allegany,301.0,Flintstone Elementary,English Language Arts 03,All Students,34,21,61.8,<= 5.0,35.3,52.9,8.8,20250808.0
1,2025,1.0,Allegany,301.0,Flintstone Elementary,English Language Arts 04,All Students,29,18,62.1,6.9,31.0,51.7,10.3,20250808.0
2,2025,1.0,Allegany,301.0,Flintstone Elementary,English Language Arts 05,All Students,23,13,56.5,13.0,30.4,52.2,<= 5.0,20250808.0
3,2025,1.0,Allegany,301.0,Flintstone Elementary,English Language Arts All Grades,All Students,86,52,60.5,7.0,32.6,52.3,8.1,20250808.0
4,2025,1.0,Allegany,401.0,South Penn Elementary,English Language Arts 03,All Students,67,20,29.9,11.9,58.2,26.9,<= 5.0,20250808.0


,Academic Year,LEA Number,LEA Name,School Number,School Name,School Type,Attend Rate Pct,Days Attended Cnt,Days Member Cnt,Fewer 5 Pct,Fewer 5 Cnt,More 20 Pct,More 20 Cnt,More 90 Member Cnt,Chronic Absentee Cnt,Chronic Absentee Denom,Chronic Absentee Pct,Create Date
0,2025,01,Allegany,0301,Flintstone Elementary,Elementary,93.2,27174.0,29158.0,23.2,38,14.6,24,164,41,208,19.7,20250924
1,2025,01,Allegany,0401,South Penn Elementary,Elementary,89.5,54333.5,60721.0,11.9,40,32.1,108,336,193,440,43.9,20250924
2,2025,01,Allegany,0402,John Humbird Elementary,Elementary,90.4,28121.0,31102.0,14.4,25,30.5,53,174,87,228,38.2,20250924
3,2025,01,Allegany,0405,Fort Hill High,High,86.2,99164.0,115069.0,14.3,92,38.0,245,645,313,690,45.4,20250924
4,2025,01,Allegany,0406,Washington Middle,Middle,88.4,90355.0,102199.0,14.4,83,34.6,199,575,260,608,42.8,20250924


,Academic Year,LEA Number,LEA Name,School Number,School Name,School Type,Ada504 Pct,Ada504 Cnt,FARMS Pct,FARMS Cnt,...,Foster Care Pct,Foster Care Cnt,Military Connected Pct,Military Connected Cnt,Economically Disadvantaged Pct,Economically Disadvantaged Cnt,Gifted Talented Pct,Gifted Talented Cnt,Total Student Cnt,Create Date
0,2025,01,Allegany,0301,Flintstone Elementary,Elementary,*,*,56.2,122,...,*,*,*,*,56.2,122,5.5,12,217,20250703
1,2025,01,Allegany,0401,South Penn Elementary,Elementary,*,*,82.3,394,...,*,*,*,*,82.3,394,*,*,479,20250703
2,2025,01,Allegany,0402,John Humbird Elementary,Elementary,*,*,88.8,221,...,*,*,*,*,88.8,221,*,*,249,20250703
3,2025,01,Allegany,0405,Fort Hill High,High,<= 5.0,*,55.5,360,...,*,*,*,*,55.2,358,28.7,186,649,20250703
4,2025,01,Allegany,0406,Washington Middle,Middle,<= 5.0,*,65.4,376,...,*,*,*,*,64.9,373,22.6,130,575,20250703


## Standardize Variable Names  
Variables of interest are selected from each source and renamed to a consistent naming convention to simplify integration across datasets and years.

In [9]:
# look at columns in each data file
print("MATH")
print(math_2025.columns.tolist())

print("\nELA")
print(ela_2025.columns.tolist())

print("\nATTENDANCE")
print(attendance_2025.columns.tolist())

print("\nSPECIAL SERVICES")
print(special_2025.columns.tolist())

MATH
['Year', 'LEA', 'LEA Name', 'School', 'School Name', 'Assessment', 'Student Group', 'Tested Count', 'Proficient Count', 'Proficient Pct', 'Level 1 Pct', 'Level 2 Pct', 'Level 3 Pct', 'Level 4 Pct', 'Create Date']

ELA
['Year', 'LEA', 'LEA Name', 'School', 'School Name', 'Assessment', 'Student Group', 'Tested Count', 'Proficient Count', 'Proficient Pct', 'Level 1 Pct', 'Level 2 Pct', 'Level 3 Pct', 'Level 4 Pct', 'Create Date']

ATTENDANCE
['Academic Year', 'LEA Number', 'LEA Name', 'School Number', 'School Name', 'School Type', 'Attend Rate Pct', 'Days Attended Cnt', 'Days Member Cnt', 'Fewer 5 Pct', 'Fewer 5 Cnt', 'More 20 Pct', 'More 20 Cnt', 'More 90 Member Cnt', 'Chronic Absentee Cnt', 'Chronic Absentee Denom', 'Chronic Absentee Pct', 'Create Date']

SPECIAL SERVICES
['Academic Year', 'LEA Number', 'LEA Name', 'School Number', 'School Name', 'School Type', 'Ada504 Pct', 'Ada504 Cnt', 'FARMS Pct', 'FARMS Cnt', 'Multilingual Learner Pct', 'Multilingual Learner Cnt', 'Migrant Pct

In [10]:
mcap_cols = {
    "Year": "year",
    "LEA": "lea_id",
    "LEA Name": "lea_name",
    "School": "school_id",
    "School Name": "school_name",
    "Assessment": "assessment",
    "Tested Count": "tested_count",
    "Proficient Count": "proficient_count",
    "Proficient Pct": "proficient_pct",
    "Level 1 Pct": "level1_pct",
    "Level 2 Pct": "level2_pct",
    "Level 3 Pct": "level3_pct",
    "Level 4 Pct": "level4_pct"
}

In [11]:
attendance_cols = {
    "Academic Year": "year",
    "LEA Number": "lea_id",
    "LEA Name": "lea_name",
    "School Number": "school_id",
    "School Name": "school_name",
    "School Type": "school_type",
    "Attend Rate Pct": "attendance_pct",
    "More 20 Pct": "more_20_pct",
    "Chronic Absentee Pct": "chronic_absentee_pct"
}

In [12]:
special_cols = {
    "Academic Year": "year",
    "LEA Number": "lea_id",
    "LEA Name": "lea_name",
    "School Number": "school_id",
    "School Name": "school_name",
    "School Type": "school_type",
    "FARMS Pct": "farms_pct",
    "Multilingual Learner Pct": "multilingual_learner_pct",
    "Students with Disabilities Pct": "students_with_disabilities_pct",
    "Economically Disadvantaged Pct": "economically_disadvantaged_pct",
    "Gifted Talented Pct": "gifted_talented_pct",
    "Total Student Cnt": "total_student_count"
}

In [13]:
def select_and_rename(df, column_map):
    return (
        df[list(column_map.keys())]
        .rename(columns=column_map)
        .copy()
    )

In [14]:
math_2025_clean = select_and_rename(math_2025, mcap_cols)
ela_2025_clean = select_and_rename(ela_2025, mcap_cols)
attendance_2025_clean = select_and_rename(attendance_2025, attendance_cols)
special_2025_clean = select_and_rename(special_2025, special_cols)

In [15]:
math_2025_clean["subject"] = "Math"
ela_2025_clean["subject"] = "ELA"

### Remove Aggregate Attendance and Special Services Records

The Attendance and Special Services files contain both individual-school records and aggregate records representing groups of schools within an LEA or statewide. Because the analysis focuses on individual schools, aggregate records are excluded. Multiple `school_type` records for the same individual school are retained because they represent distinct grade-group populations within combined schools.

In [16]:
# inspect aggregate records before standardizing IDs 
attendance_aggregate_mask = ~(
    attendance_2025_clean["school_id"]
    .astype("string")
    .str.fullmatch(r"\d+", na=False)
)

special_aggregate_mask = ~(
    special_2025_clean["school_id"]
    .astype("string")
    .str.fullmatch(r"\d+", na=False)
)

print("Attendance aggregate records:", attendance_aggregate_mask.sum())
print("Special Services aggregate records:", special_aggregate_mask.sum())

Attendance aggregate records: 103
Special Services aggregate records: 103


In [17]:
display(
    attendance_2025_clean.loc[
        attendance_aggregate_mask,
        ["lea_name", "school_id", "school_name", "school_type"]
    ].head(5)
)

display(
    special_2025_clean.loc[
        special_aggregate_mask,
        ["lea_name", "school_id", "school_name", "school_type"]
    ].tail(5)
)

,lea_name,school_id,school_name,school_type
22,Allegany,A,All Allegany Schools,Elementary
23,Allegany,A,All Allegany Schools,Middle
24,Allegany,A,All Allegany Schools,High
25,Allegany,A,All Allegany Schools,All
173,Anne Arundel,A,All Anne Arundel Schools,Elementary


,lea_name,school_id,school_name,school_type
1821,SEED,A,All SEED Schools,All
1822,State,A,All Maryland Schools,Elementary
1823,State,A,All Maryland Schools,Middle
1824,State,A,All Maryland Schools,High
1825,State,A,All Maryland Schools,All


In [18]:
def remove_aggregate_records(df):
    df = df.copy()

    # individual-school IDs contain only digits.
    # nonnumeric IDs correspond to aggregate records.
    individual_school = (
        df["school_id"]
        .astype("string")
        .str.fullmatch(r"\d+", na=False)
    )

    return df[individual_school].copy()

In [19]:
attendance_2025_clean = remove_aggregate_records(attendance_2025_clean)
special_2025_clean = remove_aggregate_records(special_2025_clean)

In [20]:
print("Attendance rows remaining:", len(attendance_2025_clean))
print("Special Services rows remaining:", len(special_2025_clean))

Attendance rows remaining: 1713
Special Services rows remaining: 1723


### Standardize School and LEA Identifiers

The source files store school and LEA identifiers differently. MCAP files load some IDs as numeric values (for example, `301.0`), while Attendance and Special Services preserve leading zeros (for example, `0301`). Identifiers are standardized as zero-padded strings, with two characters for LEA IDs and four characters for school IDs, before combining datasets.

In [21]:
# Inspect identifier formats before standardization
for name, df in {
    "Math": math_2025_clean,
    "ELA": ela_2025_clean,
    "Attendance": attendance_2025_clean,
    "Special Services": special_2025_clean
}.items():
    print(f"\n{name}")
    print(df[["lea_id", "school_id"]].head())
    print(df[["lea_id", "school_id"]].dtypes)



Math
   lea_id  school_id
0     1.0      301.0
1     1.0      301.0
2     1.0      301.0
3     1.0      301.0
4     1.0      401.0
lea_id       float64
school_id    float64
dtype: object

ELA
   lea_id  school_id
0     1.0      301.0
1     1.0      301.0
2     1.0      301.0
3     1.0      301.0
4     1.0      401.0
lea_id       float64
school_id    float64
dtype: object

Attendance
  lea_id school_id
0     01      0301
1     01      0401
2     01      0402
3     01      0405
4     01      0406
lea_id       str
school_id    str
dtype: object

Special Services
  lea_id school_id
0     01      0301
1     01      0401
2     01      0402
3     01      0405
4     01      0406
lea_id       str
school_id    str
dtype: object


In [22]:
def standardize_ids(df):
    df = df.copy()

    df["lea_id"] = (
        pd.to_numeric(df["lea_id"], errors="coerce")
        .astype("Int64")
        .astype("string")
        .str.zfill(2)
    )

    df["school_id"] = (
        pd.to_numeric(df["school_id"], errors="coerce")
        .astype("Int64")
        .astype("string")
        .str.zfill(4)
    )

    return df

In [23]:
math_2025_clean = standardize_ids(math_2025_clean)
ela_2025_clean = standardize_ids(ela_2025_clean)
attendance_2025_clean = standardize_ids(attendance_2025_clean)
special_2025_clean = standardize_ids(special_2025_clean)

In [24]:
# verify identifier formatting after standardization
for name, df in {
    "Math": math_2025_clean,
    "ELA": ela_2025_clean,
    "Attendance": attendance_2025_clean,
    "Special Services": special_2025_clean
}.items():
    print(
        f"{name}: "
        f"LEA ID lengths = {df['lea_id'].str.len().value_counts().to_dict()}, "
        f"School ID lengths = {df['school_id'].str.len().value_counts().to_dict()}"
    )

Math: LEA ID lengths = {np.int64(2): 5941}, School ID lengths = {np.int64(4): 5941}
ELA: LEA ID lengths = {np.int64(2): 5252}, School ID lengths = {np.int64(4): 5252}
Attendance: LEA ID lengths = {np.int64(2): 1713}, School ID lengths = {np.int64(4): 1713}
Special Services: LEA ID lengths = {np.int64(2): 1723}, School ID lengths = {np.int64(4): 1723}


### Inspect MCAP Assessment Structure

MCAP mathematics includes both grade-specific assessments and course-based assessments such as Algebra I, Geometry, and Algebra II, while ELA is primarily grade-based. Assessment categories are retained during initial cleaning rather than filtering solely by assessment name because some middle-school students take course-based mathematics assessments.

The final analysis will focus on elementary and middle school performance (grades 3–8). High-school records and aggregate "All Grades" results will be addressed when the analysis sample is defined.

In [25]:
print("MATH ASSESSMENTS")
print(math_2025_clean["assessment"].value_counts())

print("\nELA ASSESSMENTS")
print(ela_2025_clean["assessment"].value_counts())

MATH ASSESSMENTS
assessment
 Mathematics All Grades    1335
Mathematics 04              874
Mathematics 03              871
Mathematics 05              870
Algebra 1                   511
Mathematics 07              352
Mathematics 06              350
Mathematics 08              349
Geometry                    270
Algebra 2                   159
Name: count, dtype: int64

ELA ASSESSMENTS
assessment
 English Language Arts All Grades    1338
English Language Arts 04              874
English Language Arts 03              871
English Language Arts 05              870
English Language Arts 08              353
English Language Arts 07              352
English Language Arts 06              351
English Language Arts 10              243
Name: count, dtype: int64


### Remove Non-Data MCAP Footer Rows

The MCAP school-level worksheets contain non-data footer rows (for example, “END OF SHEET. Continued on the next sheet.”). Rows without a valid numeric year are removed, and the remaining year values are stored as integers.

In [26]:
print("Math:", math_2025_clean["year"].unique())
print("ELA:", ela_2025_clean["year"].unique())
print("Attendance:", attendance_2025_clean["year"].unique())
print("Special Services:", special_2025_clean["year"].unique())

Math: [2025 'END OF SHEET. Continued on the next sheet.']
ELA: [2025 'END OF SHEET. Continued on the next sheet.']
Attendance: [2025]
Special Services: [2025]


In [27]:
def clean_mcap_year(df):
    df = df.copy()

    df = df[
        pd.to_numeric(df["year"], errors="coerce").notna()
    ].copy()

    df["year"] = df["year"].astype(int)

    return df

In [28]:
math_2025_clean = clean_mcap_year(math_2025_clean)
ela_2025_clean = clean_mcap_year(ela_2025_clean)

In [29]:
# check that non-data row is removed
print("Math:", math_2025_clean["year"].unique())
print("ELA:", ela_2025_clean["year"].unique())

Math: [2025]
ELA: [2025]


## Clean MCAP Proficiency Values
MSDE applies disclosure-protection rules to MCAP results. Percentages at or below 5% are reported as `<= 5.0`, percentages at or above 95% as `>= 95.0`, and some results are fully suppressed as `*` under minimum sample-size requirements.

For numerical analysis, bounded values are represented using interval midpoints: `<= 5.0` is assigned 2.5% and `>= 95.0` is assigned 97.5%. Fully suppressed values are treated as missing. The original reported value is retained, and a separate status variable records whether each observation is exact, bounded, suppressed, or missing.

MSDE suppression documentation: https://reportcard.msde.maryland.gov/HelpGuides/Data%20Suppression%20Requirements%202026(A11y)CK.pdf

In [30]:
math_2025_clean["proficient_pct"].value_counts(dropna=False).head(10)

proficient_pct
<= 5.0    797
*         226
50.0       35
33.3       33
16.7       30
9.1        29
15.4       29
25.0       27
20.0       27
6.3        26
Name: count, dtype: int64

In [31]:
def clean_proficiency(df):
    df = df.copy()

    # create numeric analysis variable
    df["proficient_pct_numeric"] = pd.to_numeric(
        df["proficient_pct"],
        errors="coerce"
    )

    # replace bounded values with interval midpoints
    df.loc[
        df["proficient_pct"] == "<= 5.0",
        "proficient_pct_numeric"
    ] = 2.5

    df.loc[
        df["proficient_pct"] == ">= 95.0",
        "proficient_pct_numeric"
    ] = 97.5

    # track whether each value was reported exactly, bounded, suppressed, or missing
    df["proficient_pct_status"] = "exact"

    df.loc[
        df["proficient_pct"] == "<= 5.0",
        "proficient_pct_status"
    ] = "low_bounded"

    df.loc[
        df["proficient_pct"] == ">= 95.0",
        "proficient_pct_status"
    ] = "high_bounded"

    df.loc[
        df["proficient_pct"].eq("*"),
        "proficient_pct_status"
    ] = "suppressed"
    
    df.loc[
        df["proficient_pct"].isna(),
        "proficient_pct_status"
    ] = "missing"

    return df

In [32]:
math_2025_clean = clean_proficiency(math_2025_clean)
ela_2025_clean = clean_proficiency(ela_2025_clean)

In [33]:
for name, df in {
    "Math": math_2025_clean,
    "ELA": ela_2025_clean
}.items():
    print(f"\n{name}")
    print(df["proficient_pct_status"].value_counts(dropna=False))

    display(
        df.loc[
            df["proficient_pct"].isin(["<= 5.0", ">= 95.0", "*"]),
            ["proficient_pct", "proficient_pct_numeric", "proficient_pct_status"]
        ].drop_duplicates()
    )


Math
proficient_pct_status
exact           4901
low_bounded      797
suppressed       226
high_bounded      17
Name: count, dtype: int64


,proficient_pct,proficient_pct_numeric,proficient_pct_status
12,<= 5.0,2.5,low_bounded
13,*,NaN,suppressed
1420,>= 95.0,97.5,high_bounded



ELA
proficient_pct_status
exact           5115
suppressed        90
low_bounded       40
high_bounded       7
Name: count, dtype: int64


,proficient_pct,proficient_pct_numeric,proficient_pct_status
422,*,NaN,suppressed
438,<= 5.0,2.5,low_bounded
610,>= 95.0,97.5,high_bounded


### Clean MCAP Count Variables

Tested and proficient student counts are stored as text because suppressed values are represented by `*`. Numeric versions of these variables are created for analysis, with suppressed values converted to missing. Original reported values are retained.

In [34]:
print("MATH")
print("Tested count missing:", math_2025_clean["tested_count"].isna().sum())
print("Proficient count missing:", math_2025_clean["proficient_count"].isna().sum())

print("\nELA")
print("Tested count missing:", ela_2025_clean["tested_count"].isna().sum())
print("Proficient count missing:", ela_2025_clean["proficient_count"].isna().sum())

MATH
Tested count missing: 0
Proficient count missing: 0

ELA
Tested count missing: 0
Proficient count missing: 0


In [35]:
def clean_counts(df):
    df = df.copy()

    for col in ["tested_count", "proficient_count"]:
        df[f"{col}_numeric"] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    return df

In [36]:
math_2025_clean = clean_counts(math_2025_clean)
ela_2025_clean = clean_counts(ela_2025_clean)

In [37]:
for name, df in {
    "Math": math_2025_clean,
    "ELA": ela_2025_clean
}.items():
    print(f"\n{name}")

    for col in ["tested_count", "proficient_count"]:
        print(
            f"{col}: suppressed = {(df[col] == '*').sum()}, "
            f"numeric missing = {df[f'{col}_numeric'].isna().sum()}"
        )


Math
tested_count: suppressed = 237, numeric missing = 237
proficient_count: suppressed = 1040, numeric missing = 1040

ELA
tested_count: suppressed = 90, numeric missing = 90
proficient_count: suppressed = 137, numeric missing = 137


### Clean Attendance Percentage Variables   

MSDE applies disclosure-protection rules to reported attendance measures. Percentages at or below 5% are reported as `<= 5.0`, percentages at or above 95% as `>= 95.0`, and some results are fully suppressed as `*`.

The attendance percentage variables are cleaned using the same approach as MCAP proficiency. Exact percentages are converted to numeric values, bounded percentages are represented by interval midpoints (2.5% and 97.5%), and fully suppressed values are treated as missing. Original reported values are retained, along with status variables indicating whether each value is exact, bounded, suppressed, or missing.

In [38]:
attendance_pct_cols = [
    "attendance_pct",
    "more_20_pct",
    "chronic_absentee_pct"
]

# inspect disclosure-protected values before cleaning
for col in attendance_pct_cols:
    special_values = attendance_2025_clean.loc[
        pd.to_numeric(attendance_2025_clean[col], errors="coerce").isna(),
        col
    ].value_counts(dropna=False)

    print(f"\n{col}:")
    print(special_values.to_string(header=False))


attendance_pct:
>= 95.0    182
*           22

more_20_pct:
<= 5.0     136
*           24
>= 95.0      2

chronic_absentee_pct:
<= 5.0     28
*          13
>= 95.0     2


In [39]:
def clean_percentage(df, col):
    df = df.copy()

    numeric_col = f"{col}_numeric"
    status_col = f"{col}_status"

    # convert exact reported percentages to numeric values
    df[numeric_col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    # Use interval midpoints for bounded values
    df.loc[df[col] == "<= 5.0", numeric_col] = 2.5
    df.loc[df[col] == ">= 95.0", numeric_col] = 97.5

    # Preserve how each percentage was originally reported
    df[status_col] = "exact"
    df.loc[df[col] == "<= 5.0", status_col] = "low_bounded"
    df.loc[df[col] == ">= 95.0", status_col] = "high_bounded"
    df.loc[df[col] == "*", status_col] = "suppressed"
    df.loc[df[col].isna(), status_col] = "missing"

    return df

In [40]:
for col in attendance_pct_cols:
    attendance_2025_clean = clean_percentage(
        attendance_2025_clean,
        col
    )

In [41]:
for col in attendance_pct_cols:
    print(f"\n{col}:")
    print(
        attendance_2025_clean[f"{col}_status"]
        .value_counts(dropna=False)
        .to_string()
    )


attendance_pct:
attendance_pct_status
exact           1509
high_bounded     182
suppressed        22

more_20_pct:
more_20_pct_status
exact           1551
low_bounded      136
suppressed        24
high_bounded       2

chronic_absentee_pct:
chronic_absentee_pct_status
exact           1670
low_bounded       28
suppressed        13
high_bounded       2


## Clean Special Services Variables

Selected Special Services percentage variables are cleaned using the same disclosure-protection rules applied to other MSDE percentage measures. Exact percentages are converted to numeric values, bounded values are represented by interval midpoints, and fully suppressed (`*`) values are treated as missing. Original reported values and reporting-status variables are retained.

`total_student_count` is also converted to a numeric variable. Suppressed counts are treated as missing because no numerical range is available for substitution.

In [42]:
special_pct_cols = [
    "farms_pct",
    "multilingual_learner_pct",
    "students_with_disabilities_pct",
    "economically_disadvantaged_pct",
    "gifted_talented_pct"
]

# inspect disclosure-protected percentage values
for col in special_pct_cols:
    special_values = special_2025_clean.loc[
        pd.to_numeric(special_2025_clean[col], errors="coerce").isna(),
        col
    ].value_counts(dropna=False)

    print(f"\n{col}:")
    print(special_values.to_string(header=False))


farms_pct:
>= 95.0    320
*           33
<= 5.0       5

multilingual_learner_pct:
*         506
<= 5.0    299

students_with_disabilities_pct:
>= 95.0    82
*          49
<= 5.0     20

economically_disadvantaged_pct:
*          39
>= 95.0     6
<= 5.0      5

gifted_talented_pct:
*          304
<= 5.0     257
>= 95.0      4


In [43]:
# inspect nonnumeric total student counts
special_2025_clean.loc[
    pd.to_numeric(
        special_2025_clean["total_student_count"],
        errors="coerce"
    ).isna(),
    "total_student_count"
].value_counts(dropna=False)

total_student_count
*    50
Name: count, dtype: int64

In [44]:
for col in special_pct_cols:
    special_2025_clean = clean_percentage(
        special_2025_clean,
        col
    )

special_2025_clean["total_student_count_numeric"] = pd.to_numeric(
    special_2025_clean["total_student_count"],
    errors="coerce"
)

In [45]:
for col in special_pct_cols:
    print(f"\n{col}:")
    print(
        special_2025_clean[f"{col}_status"]
        .value_counts(dropna=False)
        .to_string()
    )

print(
    "\nTotal student count:",
    "suppressed =",
    (special_2025_clean["total_student_count"] == "*").sum(),
    "| numeric missing =",
    special_2025_clean["total_student_count_numeric"].isna().sum()
)


farms_pct:
farms_pct_status
exact           1365
high_bounded     320
suppressed        33
low_bounded        5

multilingual_learner_pct:
multilingual_learner_pct_status
exact          918
suppressed     506
low_bounded    299

students_with_disabilities_pct:
students_with_disabilities_pct_status
exact           1572
high_bounded      82
suppressed        49
low_bounded       20

economically_disadvantaged_pct:
economically_disadvantaged_pct_status
exact           1673
suppressed        39
high_bounded       6
low_bounded        5

gifted_talented_pct:
gifted_talented_pct_status
exact           1158
suppressed       304
low_bounded      257
high_bounded       4

Total student count: suppressed = 50 | numeric missing = 50


## Validate Cleaned 2025 Data

Final validation checks were performed after cleaning to confirm that identifiers and expected record keys are consistent, numeric variables contain plausible values, and school identifiers correspond to consistent school names across data sources.

### Validate IDs and Record Uniqueness

In [46]:
duplicate_checks = {
    "Math": (
        math_2025_clean,
        ["year", "lea_id", "school_id", "assessment"]
    ),
    "ELA": (
        ela_2025_clean,
        ["year", "lea_id", "school_id", "assessment"]
    ),
    "Attendance": (
        attendance_2025_clean,
        ["year", "lea_id", "school_id", "school_type"]
    ),
    "Special Services": (
        special_2025_clean,
        ["year", "lea_id", "school_id", "school_type"]
    )
}

for name, (df, keys) in duplicate_checks.items():
    print(
        f"{name}: "
        f"rows = {len(df)}, "
        f"missing LEA IDs = {df['lea_id'].isna().sum()}, "
        f"missing school IDs = {df['school_id'].isna().sum()}, "
        f"duplicate keys = {df.duplicated(subset=keys).sum()}"
    )

Math: rows = 5941, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
ELA: rows = 5252, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
Attendance: rows = 1713, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
Special Services: rows = 1723, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0


### Validate Numeric Ranges

Numeric percentage variables were checked to confirm that all observed values fall within the valid range of 0% to 100%. Count variables were also checked for implausible values, including nonpositive and non-integer counts. These checks help identify potential conversion or data-quality issues before combining data across years and sources.

In [47]:
def check_percentage_ranges(df, cols, dataset_name):
    print(f"\n{dataset_name}")

    for col in cols:
        numeric_col = f"{col}_numeric"

        print(
            f"{col}: "
            f"min = {df[numeric_col].min()}, "
            f"max = {df[numeric_col].max()}, "
            f"outside 0–100 = "
            f"{((df[numeric_col] < 0) | (df[numeric_col] > 100)).sum()}"
        )

In [48]:
check_percentage_ranges(
    math_2025_clean,
    ["proficient_pct"],
    "Math"
)

check_percentage_ranges(
    ela_2025_clean,
    ["proficient_pct"],
    "ELA"
)

check_percentage_ranges(
    attendance_2025_clean,
    attendance_pct_cols,
    "Attendance"
)

check_percentage_ranges(
    special_2025_clean,
    special_pct_cols,
    "Special Services"
)


Math
proficient_pct: min = 2.5, max = 97.5, outside 0–100 = 0

ELA
proficient_pct: min = 2.5, max = 97.5, outside 0–100 = 0

Attendance
attendance_pct: min = 43.0, max = 97.5, outside 0–100 = 0
more_20_pct: min = 2.5, max = 97.5, outside 0–100 = 0
chronic_absentee_pct: min = 2.5, max = 97.5, outside 0–100 = 0

Special Services
farms_pct: min = 2.5, max = 97.5, outside 0–100 = 0
multilingual_learner_pct: min = 2.5, max = 86.3, outside 0–100 = 0
students_with_disabilities_pct: min = 2.5, max = 97.5, outside 0–100 = 0
economically_disadvantaged_pct: min = 2.5, max = 97.5, outside 0–100 = 0
gifted_talented_pct: min = 2.5, max = 97.5, outside 0–100 = 0


In [49]:
def check_count_ranges(df, cols, dataset_name):
    print(f"\n{dataset_name}")

    for col in cols:
        numeric_col = f"{col}_numeric"
        values = df[numeric_col].dropna()

        print(
            f"{col}: "
            f"min = {values.min()}, "
            f"max = {values.max()}, "
            f"nonpositive = {(values <= 0).sum()}, "
            f"non-integer = {(values % 1 != 0).sum()}"
        )

In [50]:
check_count_ranges(
    math_2025_clean,
    ["tested_count", "proficient_count"],
    "Math"
)

check_count_ranges(
    ela_2025_clean,
    ["tested_count", "proficient_count"],
    "ELA"
)

check_count_ranges(
    special_2025_clean,
    ["total_student_count"],
    "Special Services"
)


Math
tested_count: min = 10.0, max = 1550.0, nonpositive = 0, non-integer = 0
proficient_count: min = 1.0, max = 853.0, nonpositive = 0, non-integer = 0

ELA
tested_count: min = 10.0, max = 1561.0, nonpositive = 0, non-integer = 0
proficient_count: min = 1.0, max = 1211.0, nonpositive = 0, non-integer = 0

Special Services
total_student_count: min = 15.0, max = 3248.0, nonpositive = 0, non-integer = 0


### Validate School Name Consistency Across Data Sources

School names were compared across MCAP, Attendance, and Special Services for records sharing the same LEA and school identifiers. This check was used as a secondary validation of the identifiers that will later be used to combine datasets.

MCAP mathematics and ELA school names matched exactly for all shared school IDs. A small number of naming differences were identified between MCAP and the Attendance/Special Services files; these differences reflected formatting or naming conventions rather than conflicting school identities. Because records are joined using standardized LEA and school IDs rather than school names, these schools were retained without modification.

In [51]:
math_schools = (
    math_2025_clean[
        ["lea_id", "school_id", "school_name"]
    ]
    .drop_duplicates()
)

ela_schools = (
    ela_2025_clean[
        ["lea_id", "school_id", "school_name"]
    ]
    .drop_duplicates()
)

math_ela_names = math_schools.merge(
    ela_schools,
    on=["lea_id", "school_id"],
    how="inner",
    suffixes=("_math", "_ela")
)

math_ela_mismatches = math_ela_names[
    math_ela_names["school_name_math"]
    != math_ela_names["school_name_ela"]
]

print("Matched school IDs:", len(math_ela_names))
print("Name mismatches:", len(math_ela_mismatches))

display(math_ela_mismatches.head(10))

Matched school IDs: 1334
Name mismatches: 0


,lea_id,school_id,school_name_math,school_name_ela


In [52]:
attendance_schools = (
    attendance_2025_clean[
        ["lea_id", "school_id", "school_name"]
    ]
    .drop_duplicates()
)

math_attendance_names = math_schools.merge(
    attendance_schools,
    on=["lea_id", "school_id"],
    how="inner",
    suffixes=("_mcap", "_attendance")
)

math_attendance_mismatches = math_attendance_names[
    math_attendance_names["school_name_mcap"]
    != math_attendance_names["school_name_attendance"]
]

print("Matched school IDs:", len(math_attendance_names))
print("Exact name mismatches:", len(math_attendance_mismatches))

display(math_attendance_mismatches.head(10))

Matched school IDs: 1327
Exact name mismatches: 3


,lea_id,school_id,school_name_mcap,school_name_attendance
898,16,0622,Thurgood Marshall Middle School,Thurgood Marshall Middle
908,16,0660,Drew Freeman Middle,Drew-Freeman Middle
974,16,1352,International High School at Largo,International High school @ Largo


In [53]:
special_schools = (
    special_2025_clean[
        ["lea_id", "school_id", "school_name"]
    ]
    .drop_duplicates()
)

math_special_names = math_schools.merge(
    special_schools,
    on=["lea_id", "school_id"],
    how="inner",
    suffixes=("_mcap", "_special")
)

math_special_mismatches = math_special_names[
    math_special_names["school_name_mcap"]
    != math_special_names["school_name_special"]
]

print("Matched school IDs:", len(math_special_names))
print("Exact name mismatches:", len(math_special_mismatches))

display(math_special_mismatches.head(10))

Matched school IDs: 1328
Exact name mismatches: 3


,lea_id,school_id,school_name_mcap,school_name_special
899,16,0622,Thurgood Marshall Middle School,Thurgood Marshall Middle
909,16,0660,Drew Freeman Middle,Drew-Freeman Middle
975,16,1352,International High School at Largo,International High school @ Largo


## Load and Prepare 2023–2024 Data

The 2025 data were used to inspect source structure, identify data-quality issues, and develop reusable cleaning functions. The same workflow is now applied to the 2023 and 2024 files after first confirming that filenames and column structures are consistent across years.

In [54]:
print("2023 FILES")
list_files(DATA_2023)

print("\n2024 FILES")
list_files(DATA_2024)

2023 FILES
data\raw\2023\2023 MCAP ELA Administrative Data Report Card.xlsx
data\raw\2023\2023 MCAP MATH Administrative Data Report Card.xlsx
data\raw\2023\2023_Special_Services.csv
data\raw\2023\Attendance_2023.csv

2024 FILES
data\raw\2024\2024 MCAP ELA Administrative Data Report Card.xlsx
data\raw\2024\2024 MCAP MATH Administrative Data Report Card.xlsx
data\raw\2024\2024_Special_Services.csv
data\raw\2024\Attendance_2024.csv


In [55]:
def load_year_data(year):
    year_dir = RAW_DATA_DIR / str(year)

    math_file = year_dir / f"{year} MCAP MATH Administrative Data Report Card.xlsx"
    ela_file = year_dir / f"{year} MCAP ELA Administrative Data Report Card.xlsx"
    attendance_file = year_dir / f"Attendance_{year}.csv"
    special_file = year_dir / f"{year}_Special_Services.csv"

    math = pd.read_excel(math_file, sheet_name="School_Level")
    ela = pd.read_excel(ela_file, sheet_name="School_Level")
    attendance = pd.read_csv(attendance_file)
    special = pd.read_csv(special_file)

    return math, ela, attendance, special

In [56]:
math_2023, ela_2023, attendance_2023, special_2023 = load_year_data(2023)

math_2024, ela_2024, attendance_2024, special_2024 = load_year_data(2024)

In [57]:
for year, datasets in {
    2023: {
        "Math": math_2023,
        "ELA": ela_2023,
        "Attendance": attendance_2023,
        "Special Services": special_2023
    },
    2024: {
        "Math": math_2024,
        "ELA": ela_2024,
        "Attendance": attendance_2024,
        "Special Services": special_2024
    }
}.items():
    print(f"\n{year}")
    
    for name, df in datasets.items():
        print(f"{name}: {df.shape}")


2023
Math: (5986, 15)
ELA: (5281, 15)
Attendance: (1806, 18)
Special Services: (1832, 30)

2024
Math: (5979, 15)
ELA: (5264, 15)
Attendance: (1810, 18)
Special Services: (1830, 30)


In [58]:
comparisons = [
    ("Math", math_2023, math_2024, math_2025),
    ("ELA", ela_2023, ela_2024, ela_2025),
    ("Attendance", attendance_2023, attendance_2024, attendance_2025),
    ("Special Services", special_2023, special_2024, special_2025)
]

for name, df_2023, df_2024, df_2025 in comparisons:
    print(f"\n{name}")
    print("2023 matches 2025:", df_2023.columns.equals(df_2025.columns))
    print("2024 matches 2025:", df_2024.columns.equals(df_2025.columns))


Math
2023 matches 2025: True
2024 matches 2025: True

ELA
2023 matches 2025: True
2024 matches 2025: True

Attendance
2023 matches 2025: True
2024 matches 2025: True

Special Services
2023 matches 2025: False
2024 matches 2025: True


### Harmonize Special Services Variable Names

The 2023 Special Services file uses the term `English Learner`, while the 2024–2025 files use `Multilingual Learner` for the corresponding measure. The 2023 column names were harmonized to the newer terminology so that a consistent schema could be used across all three years.

In [59]:
print("2023-only columns:")
print(set(special_2023.columns) - set(special_2025.columns))

print("\n2025-only columns:")
print(set(special_2025.columns) - set(special_2023.columns))

2023-only columns:
{'English Learner Pct', 'English Learner Cnt'}

2025-only columns:
{'Multilingual Learner Pct', 'Multilingual Learner Cnt'}


In [60]:
special_2023 = special_2023.rename(
    columns={
        "English Learner Pct": "Multilingual Learner Pct",
        "English Learner Cnt": "Multilingual Learner Cnt"
    }
)

In [61]:
special_2023.columns.equals(special_2025.columns)

True

### Select and Standardize Variables

The same variables retained from the 2025 files are selected from the 2023 and 2024 datasets and renamed using the existing column mappings. This creates consistent variable names across years before additional cleaning is applied.

In [62]:
# 2023
math_2023_clean = select_and_rename(math_2023, mcap_cols)
ela_2023_clean = select_and_rename(ela_2023, mcap_cols)
attendance_2023_clean = select_and_rename(attendance_2023, attendance_cols)
special_2023_clean = select_and_rename(special_2023, special_cols)

# 2024
math_2024_clean = select_and_rename(math_2024, mcap_cols)
ela_2024_clean = select_and_rename(ela_2024, mcap_cols)
attendance_2024_clean = select_and_rename(attendance_2024, attendance_cols)
special_2024_clean = select_and_rename(special_2024, special_cols)

In [63]:
# add subject variables
math_2023_clean["subject"] = "Math"
ela_2023_clean["subject"] = "ELA"

math_2024_clean["subject"] = "Math"
ela_2024_clean["subject"] = "ELA"

In [64]:
for name, df in {
    "2023 Math": math_2023_clean,
    "2023 ELA": ela_2023_clean,
    "2023 Attendance": attendance_2023_clean,
    "2023 Special Services": special_2023_clean,
    "2024 Math": math_2024_clean,
    "2024 ELA": ela_2024_clean,
    "2024 Attendance": attendance_2024_clean,
    "2024 Special Services": special_2024_clean
}.items():
    print(f"{name}: {df.shape}")

2023 Math: (5986, 14)
2023 ELA: (5281, 14)
2023 Attendance: (1806, 9)
2023 Special Services: (1832, 12)
2024 Math: (5979, 14)
2024 ELA: (5264, 14)
2024 Attendance: (1810, 9)
2024 Special Services: (1830, 12)


In [65]:
# remove aggregate records before standardizing IDs
attendance_2023_clean = remove_aggregate_records(attendance_2023_clean)
special_2023_clean = remove_aggregate_records(special_2023_clean)

attendance_2024_clean = remove_aggregate_records(attendance_2024_clean)
special_2024_clean = remove_aggregate_records(special_2024_clean)

In [66]:
print("2023 Attendance rows:", len(attendance_2023_clean))
print("2023 Special Services rows:", len(special_2023_clean))
print("2024 Attendance rows:", len(attendance_2024_clean))
print("2024 Special Services rows:", len(special_2024_clean))

2023 Attendance rows: 1703
2023 Special Services rows: 1729
2024 Attendance rows: 1707
2024 Special Services rows: 1727


In [67]:
for name, df in {
    "2023 Attendance": attendance_2023_clean,
    "2023 Special Services": special_2023_clean,
    "2024 Attendance": attendance_2024_clean,
    "2024 Special Services": special_2024_clean
}.items():
    print(
        f"{name}: nonnumeric school IDs =",
        (~df["school_id"].astype("string").str.fullmatch(r"\d+", na=False)).sum()
    )

2023 Attendance: nonnumeric school IDs = 0
2023 Special Services: nonnumeric school IDs = 0
2024 Attendance: nonnumeric school IDs = 0
2024 Special Services: nonnumeric school IDs = 0


In [68]:
math_2023_clean = standardize_ids(math_2023_clean)
ela_2023_clean = standardize_ids(ela_2023_clean)
attendance_2023_clean = standardize_ids(attendance_2023_clean)
special_2023_clean = standardize_ids(special_2023_clean)

math_2024_clean = standardize_ids(math_2024_clean)
ela_2024_clean = standardize_ids(ela_2024_clean)
attendance_2024_clean = standardize_ids(attendance_2024_clean)
special_2024_clean = standardize_ids(special_2024_clean)

In [69]:
for name, df in {
    "2023 Math": math_2023_clean,
    "2023 ELA": ela_2023_clean,
    "2023 Attendance": attendance_2023_clean,
    "2023 Special Services": special_2023_clean,
    "2024 Math": math_2024_clean,
    "2024 ELA": ela_2024_clean,
    "2024 Attendance": attendance_2024_clean,
    "2024 Special Services": special_2024_clean
}.items():
    lea_valid = df["lea_id"].str.fullmatch(r"\d{2}", na=False).all()
    school_valid = df["school_id"].str.fullmatch(r"\d{4}", na=False).all()

    print(
        f"{name}: "
        f"LEA IDs valid = {lea_valid}, "
        f"School IDs valid = {school_valid}"
    )

2023 Math: LEA IDs valid = True, School IDs valid = True
2023 ELA: LEA IDs valid = True, School IDs valid = True
2023 Attendance: LEA IDs valid = True, School IDs valid = True
2023 Special Services: LEA IDs valid = True, School IDs valid = True
2024 Math: LEA IDs valid = True, School IDs valid = True
2024 ELA: LEA IDs valid = True, School IDs valid = True
2024 Attendance: LEA IDs valid = True, School IDs valid = True
2024 Special Services: LEA IDs valid = True, School IDs valid = True


In [70]:
# check for non-data rows in year column
for name, df in {
    "2023 Math": math_2023_clean,
    "2023 ELA": ela_2023_clean,
    "2024 Math": math_2024_clean,
    "2024 ELA": ela_2024_clean
}.items():
    print(f"{name}: {df['year'].unique()}")

2023 Math: [2023]
2023 ELA: [2023]
2024 Math: [2024]
2024 ELA: [2024]


In [71]:
# inspect nonnumeric proficiency values
for name, df in {
    "2023 Math": math_2023_clean,
    "2023 ELA": ela_2023_clean,
    "2024 Math": math_2024_clean,
    "2024 ELA": ela_2024_clean
}.items():
    special_values = df.loc[
        pd.to_numeric(df["proficient_pct"], errors="coerce").isna(),
        "proficient_pct"
    ].value_counts(dropna=False)

    print(f"\n{name}")
    print(special_values.to_string())


2023 Math
proficient_pct
<= 5.0     1146
*           249
>= 95.0      10

2023 ELA
proficient_pct
*          95
<= 5.0     89
>= 95.0    10

2024 Math
proficient_pct
<= 5.0     1018
*           225
>= 95.0      13

2024 ELA
proficient_pct
*          88
<= 5.0     66
>= 95.0    11


In [72]:
math_2023_clean = clean_proficiency(math_2023_clean)
ela_2023_clean = clean_proficiency(ela_2023_clean)

math_2024_clean = clean_proficiency(math_2024_clean)
ela_2024_clean = clean_proficiency(ela_2024_clean)

In [73]:
# inspect nonnumeric MCAP count values
for name, df in {
    "2023 Math": math_2023_clean,
    "2023 ELA": ela_2023_clean,
    "2024 Math": math_2024_clean,
    "2024 ELA": ela_2024_clean
}.items():
    print(f"\n{name}")

    for col in ["tested_count", "proficient_count"]:
        special_values = df.loc[
            pd.to_numeric(df[col], errors="coerce").isna(),
            col
        ].value_counts(dropna=False)

        print(f"{col}:")
        print(special_values.to_string())


2023 Math
tested_count:
tested_count
*    256
proficient_count:
proficient_count
*    1405

2023 ELA
tested_count:
tested_count
*    96
proficient_count:
proficient_count
*    194

2024 Math
tested_count:
tested_count
*    236
proficient_count:
proficient_count
*    1256

2024 ELA
tested_count:
tested_count
*    90
proficient_count:
proficient_count
*    165


In [74]:
math_2023_clean = clean_counts(math_2023_clean)
ela_2023_clean = clean_counts(ela_2023_clean)

math_2024_clean = clean_counts(math_2024_clean)
ela_2024_clean = clean_counts(ela_2024_clean)

In [75]:
for col in attendance_pct_cols:
    attendance_2023_clean = clean_percentage(
        attendance_2023_clean,
        col
    )

    attendance_2024_clean = clean_percentage(
        attendance_2024_clean,
        col
    )

In [76]:
for col in special_pct_cols:
    special_2023_clean = clean_percentage(special_2023_clean, col)
    special_2024_clean = clean_percentage(special_2024_clean, col)

In [77]:
special_2023_clean["total_student_count_numeric"] = pd.to_numeric(
    special_2023_clean["total_student_count"],
    errors="coerce"
)

special_2024_clean["total_student_count_numeric"] = pd.to_numeric(
    special_2024_clean["total_student_count"],
    errors="coerce"
)

### Clean Assessment Names

Leading and trailing whitespace is removed from assessment names to ensure consistent filtering and matching.

In [78]:
for df in [
    math_2023_clean, ela_2023_clean,
    math_2024_clean, ela_2024_clean,
    math_2025_clean, ela_2025_clean
]:
    df["assessment"] = df["assessment"].str.strip()

### Validate IDs and Record Uniqueness

In [79]:
validation_specs = {
    "2023 Math": (math_2023_clean, ["year", "lea_id", "school_id", "assessment"]),
    "2023 ELA": (ela_2023_clean, ["year", "lea_id", "school_id", "assessment"]),
    "2023 Attendance": (attendance_2023_clean, ["year", "lea_id", "school_id", "school_type"]),
    "2023 Special Services": (special_2023_clean, ["year", "lea_id", "school_id", "school_type"]),
    "2024 Math": (math_2024_clean, ["year", "lea_id", "school_id", "assessment"]),
    "2024 ELA": (ela_2024_clean, ["year", "lea_id", "school_id", "assessment"]),
    "2024 Attendance": (attendance_2024_clean, ["year", "lea_id", "school_id", "school_type"]),
    "2024 Special Services": (special_2024_clean, ["year", "lea_id", "school_id", "school_type"])
}

for name, (df, keys) in validation_specs.items():
    print(
        f"{name}: "
        f"rows = {len(df)}, "
        f"missing LEA IDs = {df['lea_id'].isna().sum()}, "
        f"missing school IDs = {df['school_id'].isna().sum()}, "
        f"duplicate keys = {df.duplicated(subset=keys).sum()}"
    )

2023 Math: rows = 5986, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
2023 ELA: rows = 5281, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
2023 Attendance: rows = 1703, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
2023 Special Services: rows = 1729, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
2024 Math: rows = 5979, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
2024 ELA: rows = 5264, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
2024 Attendance: rows = 1707, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0
2024 Special Services: rows = 1727, missing LEA IDs = 0, missing school IDs = 0, duplicate keys = 0


In [80]:
check_percentage_ranges(
    math_2023_clean,
    ["proficient_pct"],
    "2023 Math"
)

check_percentage_ranges(
    ela_2023_clean,
    ["proficient_pct"],
    "2023 ELA"
)

check_percentage_ranges(
    math_2024_clean,
    ["proficient_pct"],
    "2024 Math"
)

check_percentage_ranges(
    ela_2024_clean,
    ["proficient_pct"],
    "2024 ELA"
)


2023 Math
proficient_pct: min = 2.5, max = 97.5, outside 0–100 = 0

2023 ELA
proficient_pct: min = 2.5, max = 97.5, outside 0–100 = 0

2024 Math
proficient_pct: min = 2.5, max = 97.5, outside 0–100 = 0

2024 ELA
proficient_pct: min = 2.5, max = 97.5, outside 0–100 = 0


In [81]:
check_percentage_ranges(
    attendance_2023_clean,
    attendance_pct_cols,
    "2023 Attendance"
)

check_percentage_ranges(
    attendance_2024_clean,
    attendance_pct_cols,
    "2024 Attendance"
)


2023 Attendance
attendance_pct: min = 33.8, max = 97.5, outside 0–100 = 0
more_20_pct: min = 2.5, max = 97.5, outside 0–100 = 0
chronic_absentee_pct: min = 2.5, max = 97.5, outside 0–100 = 0

2024 Attendance
attendance_pct: min = 44.2, max = 97.5, outside 0–100 = 0
more_20_pct: min = 2.5, max = 94.7, outside 0–100 = 0
chronic_absentee_pct: min = 2.5, max = 97.5, outside 0–100 = 0


In [82]:
check_percentage_ranges(
    special_2023_clean,
    special_pct_cols,
    "2023 Special Services"
)

check_percentage_ranges(
    special_2024_clean,
    special_pct_cols,
    "2024 Special Services"
)


2023 Special Services
farms_pct: min = 2.5, max = 97.5, outside 0–100 = 0
multilingual_learner_pct: min = 2.5, max = 88.4, outside 0–100 = 0
students_with_disabilities_pct: min = 2.5, max = 97.5, outside 0–100 = 0
economically_disadvantaged_pct: min = 2.5, max = 97.5, outside 0–100 = 0
gifted_talented_pct: min = 2.5, max = 97.5, outside 0–100 = 0

2024 Special Services
farms_pct: min = 2.5, max = 97.5, outside 0–100 = 0
multilingual_learner_pct: min = 2.5, max = 84.8, outside 0–100 = 0
students_with_disabilities_pct: min = 2.5, max = 97.5, outside 0–100 = 0
economically_disadvantaged_pct: min = 2.5, max = 97.5, outside 0–100 = 0
gifted_talented_pct: min = 2.5, max = 97.5, outside 0–100 = 0


In [83]:
check_count_ranges(
    math_2023_clean,
    ["tested_count", "proficient_count"],
    "2023 Math"
)

check_count_ranges(
    ela_2023_clean,
    ["tested_count", "proficient_count"],
    "2023 ELA"
)

check_count_ranges(
    math_2024_clean,
    ["tested_count", "proficient_count"],
    "2024 Math"
)

check_count_ranges(
    ela_2024_clean,
    ["tested_count", "proficient_count"],
    "2024 ELA"
)

check_count_ranges(
    special_2023_clean,
    ["total_student_count"],
    "2023 Special Services"
)

check_count_ranges(
    special_2024_clean,
    ["total_student_count"],
    "2024 Special Services"
)


2023 Math
tested_count: min = 10.0, max = 1700.0, nonpositive = 0, non-integer = 0
proficient_count: min = 1.0, max = 744.0, nonpositive = 0, non-integer = 0

2023 ELA
tested_count: min = 10.0, max = 1646.0, nonpositive = 0, non-integer = 0
proficient_count: min = 1.0, max = 1134.0, nonpositive = 0, non-integer = 0

2024 Math
tested_count: min = 10.0, max = 1643.0, nonpositive = 0, non-integer = 0
proficient_count: min = 1.0, max = 840.0, nonpositive = 0, non-integer = 0

2024 ELA
tested_count: min = 10.0, max = 1640.0, nonpositive = 0, non-integer = 0
proficient_count: min = 1.0, max = 1184.0, nonpositive = 0, non-integer = 0

2023 Special Services
total_student_count: min = 13.0, max = 3228.0, nonpositive = 0, non-integer = 0

2024 Special Services
total_student_count: min = 12.0, max = 3248.0, nonpositive = 0, non-integer = 0


## Combine Cleaned Data Across Years

In [84]:
math_all = pd.concat(
    [math_2023_clean, math_2024_clean, math_2025_clean],
    ignore_index=True
)
print(math_all.shape)
print(math_all["year"].value_counts().sort_index())

(17906, 18)
year
2023    5986
2024    5979
2025    5941
Name: count, dtype: int64


In [85]:
ela_all = pd.concat(
    [ela_2023_clean, ela_2024_clean, ela_2025_clean],
    ignore_index=True
)

print(ela_all.shape)
print(ela_all["year"].value_counts().sort_index())

(15797, 18)
year
2023    5281
2024    5264
2025    5252
Name: count, dtype: int64


In [86]:
attendance_all = pd.concat(
    [attendance_2023_clean, attendance_2024_clean, attendance_2025_clean],
    ignore_index=True
)

print(attendance_all.shape)
print(attendance_all["year"].value_counts().sort_index())

(5123, 15)
year
2023    1703
2024    1707
2025    1713
Name: count, dtype: int64


In [87]:
special_all = pd.concat(
    [special_2023_clean, special_2024_clean, special_2025_clean],
    ignore_index=True
)

print(special_all.shape)
print(special_all["year"].value_counts().sort_index())

(5179, 23)
year
2023    1729
2024    1727
2025    1723
Name: count, dtype: int64


In [88]:
combined_validation_specs = {
    "Math": (math_all, ["year", "lea_id", "school_id", "assessment"]),
    "ELA": (ela_all, ["year", "lea_id", "school_id", "assessment"]),
    "Attendance": (attendance_all, ["year", "lea_id", "school_id", "school_type"]),
    "Special Services": (special_all, ["year", "lea_id", "school_id", "school_type"])
}

for name, (df, keys) in combined_validation_specs.items():
    print(
        f"{name}: "
        f"rows = {len(df)}, "
        f"duplicate keys = {df.duplicated(subset=keys).sum()}"
    )

Math: rows = 17906, duplicate keys = 0
ELA: rows = 15797, duplicate keys = 0
Attendance: rows = 5123, duplicate keys = 0
Special Services: rows = 5179, duplicate keys = 0


## Save Cleaned Multi-Year Data

The cleaned and standardized 2023–2025 datasets are saved for use in subsequent data integration and analysis.

In [89]:
math_all.to_csv(
    PROCESSED_DATA_DIR / "mcap_math_2023_2025.csv",
    index=False
)

ela_all.to_csv(
    PROCESSED_DATA_DIR / "mcap_ela_2023_2025.csv",
    index=False
)

attendance_all.to_csv(
    PROCESSED_DATA_DIR / "attendance_2023_2025.csv",
    index=False
)

special_all.to_csv(
    PROCESSED_DATA_DIR / "special_services_2023_2025.csv",
    index=False
)